In [1]:
"""!sudo apt update
!sudo apt install -y cmake libomp-dev
!pip install --upgrade pip
!pip install git+https://github.com/carlosluis/stable-baselines3@fix_tests
!pip install gymnasium "numpy<2.0" pandas pandas-ta matplotlib "tensorflow<=2.16" scipy scikit-learn optuna"""

'!sudo apt update\n!sudo apt install -y cmake libomp-dev\n!pip install --upgrade pip\n!pip install git+https://github.com/carlosluis/stable-baselines3@fix_tests\n!pip install gymnasium "numpy<2.0" pandas pandas-ta matplotlib "tensorflow<=2.16" scipy scikit-learn optuna'

In [1]:
from utils.envs import load_dataset
import pandas as pd

In [18]:
df = load_dataset('utils/envs/stocks_data/CRM.csv')

In [19]:
df['Close']

0       159.50
1       161.11
2       161.78
3       162.86
4       162.60
         ...  
1253    291.37
1254    294.72
1255    298.01
1256    297.49
1257    306.90
Name: Close, Length: 1258, dtype: float64

In [20]:
X = df['Close'].pct_change()
X.iloc[0] = 0
X

0       0.000000
1       0.010094
2       0.004159
3       0.006676
4      -0.001596
          ...   
1253   -0.016970
1254    0.011497
1255    0.011163
1256   -0.001745
1257    0.031631
Name: Close, Length: 1258, dtype: float64

In [21]:
TP = 0.1      # Take Profit (10%)
SL = 0.05      # Stop Loss (5%)
W = 15    # Window size (look ahead W ticks)
MAX_OFFSET = len(df)  # Maximum offset (use the length of the data as the limit)

In [22]:
def calculate_B(i, X, TP, SL, W, MAX_OFFSET):
    t = 1
    for j in range(i + 1, min(i + W, MAX_OFFSET)):  # Loop within window size or MAX_OFFSET
        t *= (1 + X[j])
        if t - 1 >= TP or t - 1 <= -SL:
            return t - 1  # Return the profit/loss if the threshold is met
    return t - 1  # If the loop completes, return the final value of t - 1

# Step 3: Apply the function to calculate the 'B' indicator for each index
B = []
for idx, value in X.items():
    B.append(calculate_B(idx, X, TP, SL, W, MAX_OFFSET))

# Convert the result into a pandas Series (if needed)
B = pd.Series(B, index=X.index)

In [23]:
from sb3.combined_env import CombinedEnv

In [24]:
env = CombinedEnv(df=df, window_size=10, frame_bound=(100, 140), no_action_punishment=0)

In [25]:
env.signal_features[0]

array([-0.01419358,  0.6772073 , -0.71135294, -0.6478417 ,  0.9361222 ,
       -0.54439986,  0.37263373, -0.84010315, -0.1828074 , -0.61993045,
        0.30844018], dtype=float32)

In [26]:
len(env.B)

50

In [27]:
x = [x for x in env.S if x > 0]

In [28]:
x

[0.0892876040737576, 0.10249701918952583, 0.07076988921133934]